# Reported poaching incidents in Tanzania (2005-2015)

**Questions**

1. Where are the 268 reported poaching incidents concentrated?
2. How much can the dates in this dataset actually tell us?

**Data** - `data/poaching_points_cleaned.csv`: 268 georeferenced incident reports from
Tanzania, published with the book *The Data Visualization Workshop* (Packt). Each row has an
incident id, an incident date, a report-created date, a description and a latitude/longitude.

**Method** - pandas for the checks, k-means and a hexbin for the spatial view, matplotlib for
the figures. Every figure is written to `images/` so the README and the portfolio page can
reuse them without re-running anything.

In [1]:
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

os.makedirs("images", exist_ok=True)
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150, "font.size": 9,
                     "axes.titlesize": 11, "axes.spines.top": False, "axes.spines.right": False})

df = pd.read_csv("data/poaching_points_cleaned.csv")
print(df.shape)
df.head()

(268, 6)


,id_report,date_report,description,created_date,lat,lon
0,138,01/01/2005 12:00:00 AM,Poaching incident,2005/01/01 12:00:00 AM,-7.049359,34.841440
1,4,01/20/2005 12:00:00 AM,Poaching incident,2005/01/20 12:00:00 AM,-7.650840,34.480010
2,43,01/20/2005 12:00:00 AM,Poaching incident,2005/02/20 12:00:00 AM,-7.843202,34.005704
3,98,01/20/2005 12:00:00 AM,Poaching incident,2005/02/21 12:00:00 AM,-7.745846,33.948526
4,141,01/20/2005 12:00:00 AM,Poaching incident,2005/02/22 12:00:00 AM,-7.876673,33.690167


## 1. Basic checks

268 rows, 6 columns, a single incident type, no missing dates and no missing coordinates.
Two things are worth checking before plotting anything: duplicated report ids, and whether the
coordinates really sit inside Tanzania.

In [2]:
df["date_report"] = pd.to_datetime(df["date_report"])
df["created_date"] = pd.to_datetime(df["created_date"])

checks = {
    "rows": int(len(df)),
    "duplicate_id_report": int(df["id_report"].duplicated().sum()),
    "missing_dates": int(df["date_report"].isna().sum()),
    "missing_coords": int(df[["lat", "lon"]].isna().any(axis=1).sum()),
    "incident_date_range": [str(df["date_report"].min().date()), str(df["date_report"].max().date())],
    "lat_range": [round(df["lat"].min(), 3), round(df["lat"].max(), 3)],
    "lon_range": [round(df["lon"].min(), 3), round(df["lon"].max(), 3)],
    "descriptions": df["description"].value_counts().to_dict(),
}
print(json.dumps(checks, indent=2, default=str))

{
  "rows": 268,
  "duplicate_id_report": 0,
  "missing_dates": 0,
  "missing_coords": 0,
  "incident_date_range": [
    "2005-01-01",
    "2015-09-07"
  ],
  "lat_range": [
    -8.647,
    -6.926
  ],
  "lon_range": [
    33.442,
    35.401
  ],
  "descriptions": {
    "Poaching incident": 268
  }
}


## 2. Do the two date columns behave as documented?

The book describes `date_report` as the date the incident happened and `created_date` as the
date the report was created, so `created_date - date_report` should never be negative. It is
negative in most rows, which means the two columns cannot both mean what the documentation
says - a good reminder to check the data instead of trusting the data dictionary.

In [3]:
df["lag_days"] = (df["created_date"] - df["date_report"]).dt.days

lag_summary = {
    "negative_lag_rows": int((df["lag_days"] < 0).sum()),
    "negative_lag_share_pct": round(100 * (df["lag_days"] < 0).mean(), 1),
    "same_day_rows": int((df["lag_days"] == 0).sum()),
    "one_to_seven_days": int(((df["lag_days"] > 0) & (df["lag_days"] <= 7)).sum()),
    "more_than_seven_days": int((df["lag_days"] > 7).sum()),
    "median_lag_days": float(df["lag_days"].median()),
    "min_lag_days": float(df["lag_days"].min()),
    "max_lag_days": float(df["lag_days"].max()),
}
print(json.dumps(lag_summary, indent=2))

# how often the same incident date is reused (a sign of batch entry / rounded dates)
date_counts = df["date_report"].dt.strftime("%Y-%m-%d").value_counts()
ties = {
    "distinct_incident_dates": int(date_counts.size),
    "most_reused_date": str(date_counts.index[0]),
    "most_reused_count": int(date_counts.iloc[0]),
    "dates_used_ten_or_more_times": int((date_counts >= 10).sum()),
}
print(json.dumps(ties, indent=2))

{
  "negative_lag_rows": 167,
  "negative_lag_share_pct": 62.3,
  "same_day_rows": 19,
  "one_to_seven_days": 2,
  "more_than_seven_days": 80,
  "median_lag_days": -25.5,
  "min_lag_days": -1177.0,
  "max_lag_days": 245.0
}
{
  "distinct_incident_dates": 63,
  "most_reused_date": "2012-07-14",
  "most_reused_count": 9,
  "dates_used_ten_or_more_times": 0
}


In [4]:
fig, ax = plt.subplots(figsize=(7, 3.0))
bins = np.arange(-400, 260, 20)
ax.hist(df["lag_days"].clip(-400, 250), bins=bins, color="#4c72b0")
ax.axvline(0, color="#c44e52", linewidth=1.2, linestyle="--")
ax.text(6, ax.get_ylim()[1] * 0.92, "report created after the incident", fontsize=7.5, color="#c44e52")
ax.text(-395, ax.get_ylim()[1] * 0.92, "before the incident", fontsize=7.5, color="#c44e52")
ax.set_title("created_date - date_report for the 268 incidents (clipped at -400 days)")
ax.set_xlabel("Days")
ax.set_ylabel("Incidents")
fig.tight_layout()
fig.savefig("images/report_lag_distribution.png", bbox_inches="tight")
plt.show()

print("median:", df["lag_days"].median(), "days | negative:", int((df["lag_days"] < 0).sum()), "of", len(df))

median: -25.5 days | negative: 167 of 268


C:\Users\87506\AppData\Local\Temp\ipykernel_16632\649221684.py:12: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 3. When were the incidents recorded?

Incidents per year of `date_report`. The yearly counts are still useful as a description of
when reports were filed, but the 2015 bar covers only January-September and the spikes line up
with the reused dates found above, so this is a description rather than a trend.

In [5]:
df["year"] = df["date_report"].dt.year
by_year = df["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(by_year.index, by_year.values, color="#4c72b0")
for x, y in zip(by_year.index, by_year.values):
    ax.text(x, y + 0.8, str(y), ha="center", fontsize=7.5, color="#333")
ax.set_title("Recorded incidents per year (2015 covers Jan-Sep only)")
ax.set_xlabel("Year")
ax.set_ylabel("Incidents")
ax.set_xticks(list(by_year.index))
fig.tight_layout()
fig.savefig("images/incidents_per_year.png", bbox_inches="tight")
plt.show()

print(by_year.to_string())

2005    31
2006     7
2007     7
2008    20
2009    31
2010    25
2011    19
2012    46
2013    34
2014     2
2015    46


C:\Users\87506\AppData\Local\Temp\ipykernel_16632\2741283204.py:14: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 4. Where are they concentrated?

Three k-means clusters on the raw lon/lat with a fixed random seed, so the result is
reproducible. The number of clusters is a choice, not a finding - see the limitations.

In [6]:
X = df[["lon", "lat"]].to_numpy()
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
df["cluster"] = km.labels_

centres = km.cluster_centers_
sizes = df["cluster"].value_counts().sort_index()
order = sizes.sort_values(ascending=False).index  # biggest cluster first

fig, ax = plt.subplots(figsize=(6.2, 4.6))
colors = {"#c44e52": 0}
palette = ["#c44e52", "#4c72b0", "#55a868"]
for k in sorted(df["cluster"].unique()):
    part = df[df["cluster"] == k]
    ax.scatter(part["lon"], part["lat"], s=16, alpha=0.75, color=palette[k],
               label="cluster %d (n=%d, %.0f%%)" % (k, len(part), 100 * len(part) / len(df)))
ax.scatter(centres[:, 0], centres[:, 1], marker="X", s=150, edgecolor="black",
           facecolor="white", linewidth=1.2, label="centroid")
ax.set_title("Incident locations and three k-means clusters (n=268)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="datalim")
ax.legend(fontsize=7, loc="lower left")
fig.tight_layout()
fig.savefig("images/incident_clusters.png", bbox_inches="tight")
plt.show()

print("cluster sizes:", sizes.to_dict())
print("centroids (lon, lat):", [[round(c[0], 3), round(c[1], 3)] for c in centres])
print("largest cluster:", int(order[0]), "with", int(sizes[order[0]]), "incidents")

D:\jupiter\lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


cluster sizes: {0: 91, 1: 64, 2: 113}
centroids (lon, lat): [[33.909, -8.153], [34.98, -7.209], [34.492, -7.719]]
largest cluster: 2 with 113 incidents


C:\Users\87506\AppData\Local\Temp\ipykernel_16632\2568688934.py:25: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 5. Density without choosing a cluster count

A hexbin over the same points: the darker the cell, the more incidents were recorded inside it.
Unlike k-means this imposes no number of groups, so it is a useful cross-check on the picture
above.

In [7]:
fig, ax = plt.subplots(figsize=(6.2, 4.6))
hb = ax.hexbin(df["lon"], df["lat"], gridsize=22, cmap="magma_r", mincnt=1)
cb = fig.colorbar(hb, ax=ax, shrink=0.85)
cb.set_label("Incidents per cell")
ax.set_title("Incident density (hexbin, 268 incidents)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig("images/incident_density_hexbin.png", bbox_inches="tight")
plt.show()

cells = hb.get_array()
print("non-empty cells:", int((cells > 0).sum()), "| busiest cell:", int(cells.max()), "incidents")

non-empty cells: 148 | busiest cell: 6 incidents


C:\Users\87506\AppData\Local\Temp\ipykernel_16632\611820181.py:10: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 6. Robustness check: do the two findings support each other?

A reader of the first version asked the obvious follow-up: 62% of the rows have suspicious dates,
so are the hotspot rows the same rows? If the biggest cluster were mostly made of rows whose
dates cannot be trusted, the two findings would be one finding. So: split the rows into
date-consistent (`created_date >= date_report`) and date-inconsistent, keep the same three
clusters, and compare.

Filled markers are the consistent rows, open markers the inconsistent ones; the numbers next to
each centroid are the counts in that cluster for the two groups.

In [8]:
consistent = df[df["lag_days"] >= 0]
inconsistent = df[df["lag_days"] < 0]

def cluster_mix(d):
    c = d["cluster"].value_counts().sort_index()
    return {"n": int(len(d)), "counts": {int(k): int(v) for k, v in c.items()},
            "shares_pct": {int(k): round(100 * v / len(d), 1) for k, v in c.items()}}

mix = {"all": cluster_mix(df), "date_consistent": cluster_mix(consistent),
       "date_inconsistent": cluster_mix(inconsistent)}
print(json.dumps(mix, indent=2))

{
  "all": {
    "n": 268,
    "counts": {
      "0": 91,
      "1": 64,
      "2": 113
    },
    "shares_pct": {
      "0": 34.0,
      "1": 23.9,
      "2": 42.2
    }
  },
  "date_consistent": {
    "n": 101,
    "counts": {
      "0": 32,
      "1": 23,
      "2": 46
    },
    "shares_pct": {
      "0": 31.7,
      "1": 22.8,
      "2": 45.5
    }
  },
  "date_inconsistent": {
    "n": 167,
    "counts": {
      "0": 59,
      "1": 41,
      "2": 67
    },
    "shares_pct": {
      "0": 35.3,
      "1": 24.6,
      "2": 40.1
    }
  }
}


In [9]:
fig, ax = plt.subplots(figsize=(6.4, 4.8))
for k in sorted(df["cluster"].unique()):
    a = consistent[consistent["cluster"] == k]
    b = inconsistent[inconsistent["cluster"] == k]
    ax.scatter(a["lon"], a["lat"], s=20, color=palette[k], alpha=0.85,
               label="cluster %d, consistent dates (n=%d)" % (k, len(a)))
    ax.scatter(b["lon"], b["lat"], s=20, facecolors="none", edgecolors=palette[k],
               alpha=0.8, linewidths=0.7, label="cluster %d, suspicious dates (n=%d)" % (k, len(b)))
    cx, cy = centres[k]
    ax.annotate("%d / %d" % (len(a), len(b)), (cx, cy), textcoords="offset points", xytext=(2, 16),
                fontsize=7.5, color=palette[k], fontweight="bold", ha="center",
                bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.9))
ax.scatter(centres[:, 0], centres[:, 1], marker="X", s=150, edgecolor="black",
           facecolor="white", linewidth=1.2, zorder=5)
ax.set_title("Same clusters, split by date quality (annotated as consistent / suspicious)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="datalim")
ax.legend(fontsize=6.2, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3, frameon=False,
          handletextpad=0.3, columnspacing=1.0)
fig.tight_layout()
fig.savefig("images/cluster_robustness.png", bbox_inches="tight")
plt.show()

print("cluster sizes (counts):", sizes.to_dict())
print("largest cluster share, all rows: %.1f%%" % (100 * sizes[order[0]] / len(df)))
print("largest cluster share, consistent rows only: %.1f%%" % mix["date_consistent"]["shares_pct"][int(order[0])])
print("largest cluster share, suspicious rows only: %.1f%%" % mix["date_inconsistent"]["shares_pct"][int(order[0])])

cluster sizes (counts): {0: 91, 1: 64, 2: 113}
largest cluster share, all rows: 42.2%
largest cluster share, consistent rows only: 45.5%
largest cluster share, suspicious rows only: 40.1%


C:\Users\87506\AppData\Local\Temp\ipykernel_16632\3254464577.py:23: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 7. Summary and limitations

**What the data supports:** a spatial concentration (one cluster holds about 42% of the
reports) and a busiest hexbin cell; a description of when reports were recorded.

**What it does not support:** a seasonal story (the month pattern is dominated by a handful of
reused dates), a trend over time (2015 is a partial year and the dates are irregular), or any
statement about where poaching is *worst* - there is no denominator such as patrol effort or
protected-area size, so these are 268 *reports*, not a sample of poaching activity. The hexbin
also flatters itself: the busiest cell holds only 6 reports, so the colour scale magnifies small
differences (2 vs 6 looks dramatic and is not).

In [10]:
summary = {
    "incidents": int(len(df)),
    "incident_date_range": [str(df["date_report"].min().date()), str(df["date_report"].max().date())],
    "years_covered": int(df["year"].nunique()),
    "busiest_year": int(by_year.idxmax()),
    "busiest_year_count": int(by_year.max()),
    "quietest_year": int(by_year.idxmin()),
    "quietest_year_count": int(by_year.min()),
    "largest_cluster": int(order[0]),
    "largest_cluster_n": int(sizes[order[0]]),
    "largest_cluster_share_pct": round(100 * sizes[order[0]] / len(df), 1),
    "largest_cluster_share_consistent_pct": mix["date_consistent"]["shares_pct"][int(order[0])],
    "largest_cluster_share_suspicious_pct": mix["date_inconsistent"]["shares_pct"][int(order[0])],
    "rows_date_consistent": mix["date_consistent"]["n"],
    "rows_date_suspicious": mix["date_inconsistent"]["n"],
    "centroids_lonlat": [[round(float(c[0]), 3), round(float(c[1]), 3)] for c in centres],
    "busiest_hex_cell": int(cells.max()),
    "negative_lag_share_pct": lag_summary["negative_lag_share_pct"],
    "most_reused_date": ties["most_reused_date"],
    "most_reused_count": ties["most_reused_count"],
}
print(json.dumps(summary, indent=2))

{
  "incidents": 268,
  "incident_date_range": [
    "2005-01-01",
    "2015-09-07"
  ],
  "years_covered": 11,
  "busiest_year": 2012,
  "busiest_year_count": 46,
  "quietest_year": 2014,
  "quietest_year_count": 2,
  "largest_cluster": 2,
  "largest_cluster_n": 113,
  "largest_cluster_share_pct": 42.2,
  "largest_cluster_share_consistent_pct": 45.5,
  "largest_cluster_share_suspicious_pct": 40.1,
  "rows_date_consistent": 101,
  "rows_date_suspicious": 167,
  "centroids_lonlat": [
    [
      33.909,
      -8.153
    ],
    [
      34.98,
      -7.209
    ],
    [
      34.492,
      -7.719
    ]
  ],
  "busiest_hex_cell": 6,
  "negative_lag_share_pct": 62.3,
  "most_reused_date": "2012-07-14",
  "most_reused_count": 9
}
